In [74]:
import pandas as pd 
import io
from transformers import pipeline   
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os


In [46]:
def load_data(file)->pd.DataFrame: #eloutput_bytla3_dataframe
    return pd.read_json(file)

def validate_data(data:pd.DataFrame)->bool: # msh mohma awy bs bn make sure en el data tamam
    if data.empty:
        return False
    if not all(col in data.columns for col in ["url", "title", "content"]):
        return False
    return True


In [47]:
data = load_data("dataset.json")
valid = validate_data(data)
print(f"Data is valid: {valid}")

Data is valid: True


In [48]:
def clean_text(text: str) -> str:
    return text.strip()

In [49]:
data['content'] = data['content'].apply(clean_text)

In [50]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    separators=[
        "\n\n",   
        "\n",     
        ". ",     
        " ",      
        ""        
    ]
)
# hn2sm el content bta3 kol row l chunks kol chunk btgrab t separte 3nd el 7agat de

In [51]:

chunks = []

for _, row in data.iterrows():

    text_chunks = splitter.split_text(row["content"])

    for i, chunk in enumerate(text_chunks):
        chunks.append({
            "text": chunk,
            "title": row["title"],
            "url": row["url"],
            "chunk_id": i
        })


In [52]:
print(chunks[71])  

{'text': '. To decrypt the stream, the player requests the key from Media Services key delivery service or the key delivery service you specified. To decide if the user is authorized to get the key, the service evaluates the content key policy that you specified for the key. You can use the REST API, or a Media Services client library to configure authorization and authentication policies for your licenses and keys. Widevine is not available in the GovCloud region. Note Media services will be enforcing TLS 1.2 for all requests to KeyDelivery, RESTv2, Streaming Endpoint and Live Event streaming origins. Accounts with existing TLS 1.0 or 1.1 usage will be exempt from this enforcement', 'title': 'Content protection with dynamic encryption and key delivery', 'url': 'https://learn.microsoft.com/en-us/azure/media-services/latest/drm-content-protection-concept', 'chunk_id': 3}


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2') # by3ml sentence transformation 3shan tb2a vector 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [54]:
embeddings = model.encode([chunk["text"] for chunk in chunks])


In [55]:
print(embeddings.shape)  

(427, 384)


In [56]:
query = "What is microsoft azure?"

In [ ]:
query_embedding = model.encode([query]) # brdo b7olo l vector 3shan a2dr akaren b elsimilarity ba3den

In [58]:
print(query_embedding.shape)

(1, 384)


In [59]:
from sklearn.metrics.pairwise import cosine_similarity

In [60]:
score = cosine_similarity(query_embedding, embeddings)

In [61]:
print(score)

[[ 0.52127695  0.5215465   0.44103473  0.4935134   0.30033112  0.36999094
   0.5537442   0.5191537   0.557537    0.46692783  0.50759387  0.45829356
   0.3037557   0.37502843  0.29543108  0.28044254  0.18862289  0.1625646
   0.50578934  0.47635335  0.47596863  0.45471084  0.18528655  0.3016097
   0.18703601  0.2281661   0.3379408   0.4381704   0.39339522  0.4176076
   0.5585456   0.4735122   0.39420605  0.5144852   0.5207546   0.39752156
   0.43825108  0.44791678  0.48295555  0.25391626  0.55945873  0.47269762
   0.36640882  0.42200637  0.36591333  0.61285233  0.59736097  0.55005914
   0.47750744  0.42747813  0.10677023  0.5338939   0.57364154  0.23409002
   0.3902002   0.44899103  0.36347765  0.38294062  0.33311895  0.4589431
   0.39000058  0.18671274  0.41814628  0.5423184   0.3089051   0.14595851
   0.4265713   0.22108796  0.43600678  0.18467318  0.36552826  0.09592371
   0.0389916   0.32464987  0.31952143  0.23869988  0.3884671   0.03945397
   0.11698875  0.24802351  0.10580096  0.2

In [62]:
import numpy as np
best_match = np.argmax(score)
print(chunks[best_match])

{'text': '. Whether you choose to host your applications entirely in Azure or extend your on-premises applications with Azure services, Azure helps you create applications that are scalable, reliable, and maintainable. Azure supports the most popular programming languages in use today, including .NET, C++, Go, Java, JavaScript, Python, and Rust. With a comprehensive SDK and extensive support in tools you already use like VS Code, Visual Studio, IntelliJ, and Eclipse, Azure builds on the skills you already have and helps you be productive right away. Azure also provides a suite of developer tools that streamline how you build, deploy, and manage cloud applications', 'title': 'Azure for developers overview', 'url': 'https://learn.microsoft.com/en-us/azure/developer/intro/azure-developer-overview', 'chunk_id': 1}


In [63]:
import chromadb

In [ ]:
client =chromadb.Client() # de database third party 3shan t5zn feha elvectors w bt3ml embedding lw7dha

In [ ]:
collection =client.get_or_create_collection(name="azure_collection")# kany b store bs el 7aga de f database 3shan hwa b3ml embedding leh aslan 3shan ans hena b search by meaning msh zy el sql data base
docs =collection.add(documents=[chunk["text"] for chunk in chunks],
               metadatas=[{"title": chunk["title"], "url": chunk["url"], "chunk_id": chunk["chunk_id"]} for chunk in chunks],
               ids=[str(i) for i in range(len(chunks))])

In [78]:
results = collection.query(query_texts=["what is azure?"],n_results=2)

In [79]:
retrieved_docs = results["documents"][0]

In [80]:
context = "\n\n".join(retrieved_docs)

In [81]:
prompt = f"""
You are an AI assistant for Microsoft Azure documentation.

Answer the user's question using ONLY the information provided in the context below.

Rules:
- Do NOT use outside knowledge.
- Do NOT make up information.
- If the answer is not found in the context, reply:
  "I couldn't find the answer in the provided documentation."
- Give a concise and accurate answer.

Context:
{context}

Question:
{query}

Answer:
"""


In [ ]:
genai.configure(api_key="API key")

model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content(prompt)

print(response.text)

Azure helps you create applications that are scalable, reliable, and maintainable, whether you choose to host them entirely in Azure or extend your on-premises applications with Azure services.
